In [5]:
# --- 1. DEPENDENCIES & EXTENSIVE DOWNLOADS ---
!pip install -q datasets nltk textstat transformers torch lexicalrichness

import nltk
import torch
import math
import warnings
import pandas as pd
import numpy as np
from collections import Counter
from scipy import stats
from datasets import load_dataset
from itertools import islice
from nltk.tokenize import sent_tokenize, word_tokenize
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from lexicalrichness import LexicalRichness

warnings.filterwarnings("ignore")

# Download necessary NLTK packages silently
print("Downloading NLTK packages...")
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
english_stopwords = set(stopwords.words('english'))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[SUCCESS] Setup complete. Engine executing on: {device}")

[SUCCESS] Setup complete. Engine executing on: cuda


In [6]:
# --- 2. ADVANCED LINGUISTIC MATH ENGINE ---
print("Loading Perplexity Model (GPT-2)...")
eval_model_name = "gpt2"
eval_tokenizer = GPT2TokenizerFast.from_pretrained(eval_model_name)
eval_model = GPT2LMHeadModel.from_pretrained(eval_model_name).to(device)
eval_model.eval()
print("[SUCCESS] Model loaded.")

def safe_divide(n, d):
    """Prevents division by zero crashes on messy internet text."""
    return n / d if d and d > 0 else 0

def calculate_perplexity(text):
    """Predictability: Lower = mathematically chained/formulaic."""
    # THE FIX: truncation=True prevents the out-of-bounds IndexError
    encodings = eval_tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    input_ids = encodings.input_ids.to(device)
    
    if input_ids.size(1) <= 1: return np.nan
    with torch.no_grad():
        loss = eval_model(input_ids, labels=input_ids).loss
    return torch.exp(loss).item()

def run_comprehensive_suite(text):
    """Executes distinct metrics to isolate AI slopification."""
    try:
        tokens = [t.lower() for t in word_tokenize(text) if t.isalpha()]
        sentences = sent_tokenize(text)
        total_words = len(tokens)
        
        # Skip fragments that will break the math
        if total_words < 30 or len(sentences) < 3: 
            return None 
            
        tags = nltk.pos_tag(tokens) # Removed lang='eng' to prevent NLTK version errors
        tag_counts = Counter([tag for word, tag in tags])
        
        # Lexical Richness (MTLD)
        try:
            lex = LexicalRichness(text)
            mtld_score = lex.mtld(threshold=0.72)
        except:
            mtld_score = np.nan
        
        # Burstiness (Coefficient of Variation)
        sent_lengths = [len(word_tokenize(s)) for s in sentences]
        cv_burstiness = safe_divide(np.std(sent_lengths), np.mean(sent_lengths))
        
        # Stop-Word Density (Filler)
        stop_count = sum(1 for t in tokens if t in english_stopwords)
        stop_density = safe_divide(stop_count, total_words)
        
        # Adjective-to-Verb Ratio (Show vs. Tell)
        adj_count = sum(v for k, v in tag_counts.items() if k.startswith('J'))
        verb_count = sum(v for k, v in tag_counts.items() if k.startswith('V'))
        adj_verb_ratio = safe_divide(adj_count, verb_count)
        
        # Syntactic Repetition (POS Entropy)
        tag_probs = [safe_divide(count, total_words) for count in tag_counts.values()]
        pos_entropy = -sum(p * math.log2(p) for p in tag_probs if p > 0)
        
        # Trigram Uniqueness
        ngrams = [tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
        trigram_uniqueness = safe_divide(len(set(ngrams)), len(ngrams))

        return {
            "Predictability (Perplexity)": calculate_perplexity(text),
            "Diversity (MTLD)": mtld_score,
            "Burstiness (CV)": cv_burstiness,
            "Filler (StopWord %)": stop_density,
            "Lazy Writing (Adj/Verb Ratio)": adj_verb_ratio,
            "Syntactic Variance (POS Entropy)": pos_entropy,
            "Structural Uniqueness (Trigram %)": trigram_uniqueness
        }
    except Exception as e:
        return None

Loading Perplexity Model (GPT-2)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SUCCESS] Model loaded.


In [7]:
# --- 3. REPRODUCIBLE TEMPORAL STREAMING ---
print("Streaming dataset buckets from Hugging Face...")

HF_TOKEN = "YOUR_TOKEN_HERE" # REPLACE

# 2021 Human Data
stream_2021 = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
data_2021 = [x['text'][:1500] for x in islice(stream_2021, 1000) if len(x['text']) > 400]
print(f"[SUCCESS] Captured {len(data_2021)} samples for 2021 (Human).")

# 2026 AI Slop Data
stream_2026 = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True, token=HF_TOKEN)
data_2026 = []
for x in islice(stream_2026, 4000):
    reply = x['conversation'][1]['content'] if len(x['conversation']) > 1 else ""
    if len(reply) > 400:
        data_2026.append(reply[:1500])
    if len(data_2026) == 1000: break

print(f"[SUCCESS] Captured {len(data_2026)} samples for 2026 (AI Era).")

Streaming dataset buckets from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

[SUCCESS] Captured 1000 samples for 2021 (Human).


README.md: 0.00B [00:00, ?B/s]

[SUCCESS] Captured 1000 samples for 2026 (AI Era).


In [8]:
# --- 4. EXTENSIVE EVALUATION & STATISTICAL INSIGHTS ---
print("\nProcessing 2021 text through linguistic suite...")
metrics_2021 = [run_comprehensive_suite(t) for t in data_2021]
df_2021 = pd.DataFrame([m for m in metrics_2021 if m is not None])
df_2021['Era'] = '2021 (Human)'

print("Processing 2026 text through linguistic suite...")
metrics_2026 = [run_comprehensive_suite(t) for t in data_2026]
df_2026 = pd.DataFrame([m for m in metrics_2026 if m is not None])
df_2026['Era'] = '2026 (AI Era)'

df_macro = pd.concat([df_2021, df_2026]).dropna()

print("\n=======================================================================")
print("                      EXTENSIVE LINGUISTIC SUMMARY")
print("=======================================================================")
# Transpose the output to make it highly readable in the Kaggle console
print(df_macro.groupby('Era').mean().T.to_string())
print("=======================================================================")

# Calculate p-value for Perplexity to track structural predictability shifts
t_stat, p_val = stats.ttest_ind(df_2021['Predictability (Perplexity)'], df_2026['Predictability (Perplexity)'])
print(f"\n[STATISTICS] Predictability (Perplexity) Shift p-value: {p_val:.5e}")
if p_val < 0.05:
    print("CONCLUSION: The shift in structural predictability is STATISTICALLY SIGNIFICANT.")


Processing 2021 text through linguistic suite...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Processing 2026 text through linguistic suite...

                      EXTENSIVE LINGUISTIC SUMMARY
Era                                2021 (Human)  2026 (AI Era)
Predictability (Perplexity)           26.589529      17.241446
Diversity (MTLD)                     105.914852      55.856596
Burstiness (CV)                        0.517817       0.471304
Filler (StopWord %)                    0.438978       0.398613
Lazy Writing (Adj/Verb Ratio)          0.600304       0.786762
Syntactic Variance (POS Entropy)       3.645850       3.236458
Structural Uniqueness (Trigram %)      0.978320       0.905795

[STATISTICS] Predictability (Perplexity) Shift p-value: 8.52881e-23
CONCLUSION: The shift in structural predictability is STATISTICALLY SIGNIFICANT.
